# Bhojpuri GPT-2 Large Manual Fine-Tuning

This notebook is inspired by your `part2_finetune_GPT2gulliver.ipynb` approach:
- manual GPT-2 training loop
- random token-window sampling
- no Hugging Face `Trainer`
- Colab T4-friendly defaults

Recommended on T4: keep `TRAIN_MODE = 'lora'`.


In [ ]:
!pip install -q -U transformers peft sentencepiece


In [ ]:
import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), 'Enable GPU in Colab: Runtime > Change runtime type > GPU'


In [ ]:
from pathlib import Path

BASE_DIR = Path('/content/bhojpuri_gpt2_manual')
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs' / 'gpt2-large-bhojpuri-manual'

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR =', DATA_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)


## Optional: Use Google Drive

Set `USE_DRIVE = True` if you want checkpoints to survive runtime disconnects.


In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    from pathlib import Path
    BASE_DIR = Path('/content/drive/MyDrive/bhojpuri_gpt2_manual')
    DATA_DIR = BASE_DIR / 'data'
    OUTPUT_DIR = BASE_DIR / 'outputs' / 'gpt2-large-bhojpuri-manual'
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print('Using Google Drive paths')
    print('DATA_DIR =', DATA_DIR)
    print('OUTPUT_DIR =', OUTPUT_DIR)
else:
    print('Using /content paths')


## Upload Dataset Files

Upload these two files from your machine:
- `lyrics_only_train.txt`
- `lyrics_only_val.txt`

Local source folder on your machine:
- `F:\\bhojpuri\\results\\gpt2_bhojpuri`


In [ ]:
from google.colab import files
uploaded = files.upload()

required = {'lyrics_only_train.txt', 'lyrics_only_val.txt'}
for name, content in uploaded.items():
    (DATA_DIR / name).write_bytes(content)

missing = [name for name in required if not (DATA_DIR / name).exists()]
assert not missing, f'Missing required files: {missing}'

for path in sorted(DATA_DIR.iterdir()):
    print(path.name, path.stat().st_size)


## Write The Manual Training Script

This is the stable, Gulliver-style manual training loop.


In [ ]:
%%writefile train_gpt2_large_manual.py
﻿#!/usr/bin/env python
"""Manual GPT-2 Large fine-tuning loop for the Bhojpuri corpus.

This script is designed for Colab T4 stability:
- no Hugging Face Trainer
- random token-window sampling inspired by the Gulliver notebook
- optional LoRA mode for lower memory usage
- mixed precision and gradient accumulation
"""

from __future__ import annotations

import argparse
import json
import math
import random
from contextlib import nullcontext
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed



def parse_args(argv: list[str]) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description='Manual GPT-2 Large fine-tuning for Bhojpuri lyrics.')
    parser.add_argument('--model-name', default='gpt2-large')
    parser.add_argument('--train-file', type=Path, required=True, help='Plain text training corpus.')
    parser.add_argument('--validation-file', type=Path, required=True, help='Plain text validation corpus.')
    parser.add_argument('--output-dir', type=Path, required=True)
    parser.add_argument('--train-mode', choices=['lora', 'full'], default='lora')
    parser.add_argument('--seq-len', type=int, default=128)
    parser.add_argument('--batch-size', type=int, default=1)
    parser.add_argument('--gradient-accumulation-steps', type=int, default=16)
    parser.add_argument('--train-steps', type=int, default=1200)
    parser.add_argument('--eval-interval', type=int, default=100)
    parser.add_argument('--eval-batches', type=int, default=16)
    parser.add_argument('--save-interval', type=int, default=200)
    parser.add_argument('--learning-rate', type=float, default=None)
    parser.add_argument('--weight-decay', type=float, default=0.01)
    parser.add_argument('--seed', type=int, default=42)
    parser.add_argument('--lora-r', type=int, default=8)
    parser.add_argument('--lora-alpha', type=int, default=16)
    parser.add_argument('--lora-dropout', type=float, default=0.05)
    parser.add_argument('--disable-gradient-checkpointing', action='store_true')
    parser.add_argument('--resume-from', type=Path, default=None, help='Optional adapter/model checkpoint to resume from.')
    return parser.parse_args(argv)



def choose_learning_rate(args: argparse.Namespace) -> float:
    if args.learning_rate is not None:
        return args.learning_rate
    return 2e-4 if args.train_mode == 'lora' else 5e-5



def load_text(path: Path) -> str:
    text = path.read_text(encoding='utf-8')
    if not text.strip():
        raise ValueError(f'Empty text file: {path}')
    return text



def tokenize_text(tokenizer, text: str) -> torch.Tensor:
    return tokenizer.encode(text, return_tensors='pt')[0]



def build_model_and_tokenizer(args: argparse.Namespace, device: torch.device):
    tokenizer = AutoTokenizer.from_pretrained(args.model_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(args.model_name)
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False

    if args.train_mode == 'lora':
        from peft import LoraConfig, PeftModel, TaskType, get_peft_model

        if args.resume_from and (args.resume_from / 'adapter_config.json').exists():
            model = PeftModel.from_pretrained(model, args.resume_from)
        else:
            if hasattr(model, 'enable_input_require_grads'):
                model.enable_input_require_grads()
            peft_config = LoraConfig(
                task_type=TaskType.CAUSAL_LM,
                inference_mode=False,
                r=args.lora_r,
                lora_alpha=args.lora_alpha,
                lora_dropout=args.lora_dropout,
                target_modules=['c_attn', 'c_proj'],
                fan_in_fan_out=True,
                bias='none',
            )
            model = get_peft_model(model, peft_config)
    elif args.resume_from and (args.resume_from / 'config.json').exists():
        model = AutoModelForCausalLM.from_pretrained(args.resume_from)

    if not args.disable_gradient_checkpointing and hasattr(model, 'gradient_checkpointing_enable'):
        model.gradient_checkpointing_enable()

    model.to(device)
    return model, tokenizer



def get_trainable_parameters(model: torch.nn.Module) -> list[torch.nn.Parameter]:
    return [param for param in model.parameters() if param.requires_grad]



def sample_batch(tokens: torch.Tensor, batch_size: int, seq_len: int, device: torch.device) -> torch.Tensor:
    max_start = len(tokens) - seq_len - 1
    if max_start <= 0:
        raise ValueError('Token stream is shorter than seq_len + 1.')
    starts = torch.randint(max_start, (batch_size,))
    offsets = torch.arange(seq_len)
    batch = tokens[starts[:, None] + offsets]
    return batch.to(device)



def mean_loss(model, tokens: torch.Tensor, batch_size: int, seq_len: int, batches: int, device: torch.device) -> float:
    model.eval()
    losses: list[float] = []
    amp_ctx = torch.autocast(device_type='cuda', dtype=torch.float16) if device.type == 'cuda' else nullcontext()
    with torch.no_grad():
        for _ in range(batches):
            x = sample_batch(tokens, batch_size=batch_size, seq_len=seq_len, device=device)
            with amp_ctx:
                loss = model(x, labels=x).loss
            losses.append(float(loss.detach().cpu()))
    model.train()
    return sum(losses) / max(len(losses), 1)



def save_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')



def save_checkpoint(model, tokenizer, output_dir: Path, step: int, history: list[dict[str, float]]) -> None:
    ckpt_dir = output_dir / f'checkpoint-step-{step}'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(ckpt_dir)
    tokenizer.save_pretrained(ckpt_dir)
    save_json(ckpt_dir / 'history.json', {'history': history})



def main(argv: list[str]) -> int:
    args = parse_args(argv)
    args.output_dir.mkdir(parents=True, exist_ok=True)

    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        if hasattr(torch, 'set_float32_matmul_precision'):
            torch.set_float32_matmul_precision('high')

    set_seed(args.seed)
    random.seed(args.seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    train_text = load_text(args.train_file)
    val_text = load_text(args.validation_file)

    model, tokenizer = build_model_and_tokenizer(args, device=device)
    train_tokens = tokenize_text(tokenizer, train_text)
    val_tokens = tokenize_text(tokenizer, val_text)

    trainable_params = get_trainable_parameters(model)
    optimizer = torch.optim.AdamW(trainable_params, lr=choose_learning_rate(args), weight_decay=args.weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == 'cuda')
    amp_ctx = torch.autocast(device_type='cuda', dtype=torch.float16) if device.type == 'cuda' else nullcontext()

    history: list[dict[str, float]] = []
    running_loss = 0.0
    running_count = 0
    optimizer.zero_grad(set_to_none=True)
    tokens_per_step = args.batch_size * args.seq_len * args.gradient_accumulation_steps

    if args.train_mode == 'lora' and hasattr(model, 'print_trainable_parameters'):
        model.print_trainable_parameters()

    run_config = {
        'generated_at': datetime.now(timezone.utc).isoformat(),
        'model_name': args.model_name,
        'train_mode': args.train_mode,
        'train_file': str(args.train_file.resolve()),
        'validation_file': str(args.validation_file.resolve()),
        'output_dir': str(args.output_dir.resolve()),
        'device': str(device),
        'seq_len': args.seq_len,
        'batch_size': args.batch_size,
        'gradient_accumulation_steps': args.gradient_accumulation_steps,
        'train_steps': args.train_steps,
        'eval_interval': args.eval_interval,
        'eval_batches': args.eval_batches,
        'learning_rate': choose_learning_rate(args),
        'weight_decay': args.weight_decay,
        'train_tokens': int(train_tokens.numel()),
        'validation_tokens': int(val_tokens.numel()),
        'approx_tokens_per_optimizer_step': int(tokens_per_step),
        'approx_optimizer_steps_per_pass': max(1, math.ceil(train_tokens.numel() / max(tokens_per_step, 1))),
    }
    save_json(args.output_dir / 'run_config.json', run_config)

    print(json.dumps(run_config, ensure_ascii=False, indent=2))
    print('Starting training...')

    for step in range(1, args.train_steps + 1):
        x = sample_batch(train_tokens, batch_size=args.batch_size, seq_len=args.seq_len, device=device)

        with amp_ctx:
            loss = model(x, labels=x).loss
            scaled_loss = loss / args.gradient_accumulation_steps

        scaler.scale(scaled_loss).backward()
        running_loss += float(loss.detach().cpu())
        running_count += 1

        if step % args.gradient_accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        if step % 10 == 0 or step == 1:
            avg_loss = running_loss / max(running_count, 1)
            print(f'step {step:5d}/{args.train_steps} | train_loss {avg_loss:.4f}')
            if step % 10 == 0:
                running_loss = 0.0
                running_count = 0

        if step % args.eval_interval == 0 or step == args.train_steps:
            val_loss = mean_loss(
                model,
                val_tokens,
                batch_size=args.batch_size,
                seq_len=args.seq_len,
                batches=args.eval_batches,
                device=device,
            )
            record = {'step': step, 'val_loss': val_loss}
            history.append(record)
            print(f'>>> eval @ step {step}: val_loss={val_loss:.4f}')
            save_json(args.output_dir / 'history.json', {'history': history})

        if step % args.save_interval == 0 or step == args.train_steps:
            save_checkpoint(model, tokenizer, args.output_dir, step, history)

    final_dir = args.output_dir / 'final'
    final_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(final_dir)
    tokenizer.save_pretrained(final_dir)
    save_json(final_dir / 'history.json', {'history': history})
    print(f'Finished. Final model saved to {final_dir}')
    return 0


if __name__ == '__main__':
    import sys

    raise SystemExit(main(sys.argv[1:]))



In [ ]:
TRAIN_FILE = str(DATA_DIR / 'lyrics_only_train.txt')
VAL_FILE = str(DATA_DIR / 'lyrics_only_val.txt')
OUT_DIR = str(OUTPUT_DIR)

TRAIN_MODE = 'lora'
SEQ_LEN = 128
BATCH_SIZE = 1
GRAD_ACCUM = 16
TRAIN_STEPS = 1200
EVAL_INTERVAL = 100
EVAL_BATCHES = 16
SAVE_INTERVAL = 200

print('TRAIN_FILE =', TRAIN_FILE)
print('VAL_FILE =', VAL_FILE)
print('OUT_DIR =', OUT_DIR)
print('TRAIN_MODE =', TRAIN_MODE)
print('SEQ_LEN =', SEQ_LEN)
print('TRAIN_STEPS =', TRAIN_STEPS)


## Start Training

This cell avoids `Trainer`, `DataCollator`, and the API compatibility issues you were hitting earlier.


In [ ]:
!python train_gpt2_large_manual.py \
  --model-name gpt2-large \
  --train-file "$TRAIN_FILE" \
  --validation-file "$VAL_FILE" \
  --output-dir "$OUT_DIR" \
  --train-mode "$TRAIN_MODE" \
  --seq-len $SEQ_LEN \
  --batch-size $BATCH_SIZE \
  --gradient-accumulation-steps $GRAD_ACCUM \
  --train-steps $TRAIN_STEPS \
  --eval-interval $EVAL_INTERVAL \
  --eval-batches $EVAL_BATCHES \
  --save-interval $SAVE_INTERVAL


## Quick Generation Test

If you trained with LoRA, this cell loads the base GPT-2 Large model plus the saved adapter.


In [ ]:
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_DIR = Path(OUT_DIR) / 'final'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if (MODEL_DIR / 'adapter_config.json').exists():
    base_model = AutoModelForCausalLM.from_pretrained('gpt2-large')
    model = PeftModel.from_pretrained(base_model, MODEL_DIR)
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_DIR)

model.to(device)
model.eval()

prompt = 'Pyar ke baat ' 
inputs = tokenizer(prompt, return_tensors='pt').to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        top_p=0.95,
        top_k=50,
        temperature=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))


## Optional: Zip Outputs

Use this if you want a single archive to download from Colab.


In [ ]:
import shutil
archive_base = str(BASE_DIR / 'gpt2-large-bhojpuri-manual')
zip_path = shutil.make_archive(archive_base, 'zip', root_dir=OUTPUT_DIR)
print('Created:', zip_path)
